**Rehydrate/deserialize a Python dictionary from a JSON formatted text file**

In [ ]:
import json
from pathlib import Path

# Cell 01

file_name = "periodic_table.json"
with Path(file_name).open("r", encoding="utf-8") as f_in:
    periodic_table = json.load(f_in)

periodic_table["elements"][0]["name"]

**Create a Python `list` of elements sorted by group, then period, then atomic number**
1. Use `sorted()` with a key tuple to sort all elements by group, then period, then atomic number in O(n log n)
2. Use a list comprehension to build the final `list`, where each item is a <u>tuple</u> containing:\
a. A `string` label formed by its atomic symbol, a hyphen, and its atomic number\
b. The element's melting point (or $-∞$ if the melting point is not known)\
c. The element's boiling point (or $+∞$ if the boiling point is not known)

In [ ]:
import numpy as np
import pandas as pd

# Cell 02

elements: list = sorted(
    periodic_table["elements"],
    key=lambda v: (int(v["group"]), int(v["period"]), int(v["number"])),
)
elements = [
    (
        f"{v['symbol']}-{v['number']}",
        float(v["melt"]) if v["melt"] is not None else -np.inf,
        float(v["boil"]) if v["boil"] is not None else np.inf,
    )
    for v in elements
]
pd.DataFrame(elements[:10], columns=["Element", "Melt", "Boil"])

**In preperation for plotting the melting and boiling points:**
1. Create a numpy `array` from the elements `list`
2. Create a numpy `array` of the melting points, which are the 2nd item [index #1] in each tuple in the elements list
2. Create a numpy `array` of the boiling points, which are the 3rd item [index #2] in each tuple in the elements list
3. Convert both melting and boiling point values from Kelvin to Celsius

In [ ]:
# Cell 03

data = np.array(elements)
melt = np.array(data[:, 1], dtype=float) - 273.15
boil = np.array(data[:, 2], dtype=float) - 273.15
pd.DataFrame(
    {
        "Element": [element[0] for element in elements[:10]],
        "Melt": melt[:10],
        "Boil": boil[:10],
    }
)

**Find the element with the <u>smallest</u> liquid range (boiling point - melting point)**
1. The variable `liquid_range` becomes an numpy `array` because $boil$ and $melt$ are both arrays
2. The function `np.argmin()` returns the index in the given array with the minimum value

In [ ]:
# Cell 04

liquid_range = boil - melt
min_idx = np.argmin(liquid_range)
min_range = liquid_range[min_idx]
print(f"Smallest liquid range: {min_range:,.2f}°C is {data[min_idx, 0]}")

**Find the element with the <u>largest</u> liquid range**
1. Use `np.isfinite()` to mask out elements with an unknown boiling point (stored as $+∞$)
2. Use `np.where()` to substitute $-∞$ for any non-finite liquid range values
3. Use `np.argmax()` to find the index of the largest finite liquid range

In [ ]:
# Cell 05

finite_mask = np.isfinite(liquid_range)
max_idx = np.argmax(np.where(finite_mask, liquid_range, -np.inf))
max_range = liquid_range[max_idx]
print(f" Largest liquid range: {max_range:,.2f}°C is {data[max_idx, 0]}")

**Plot the melting point and boiling point (if available)\
for every element sorted by group, period, atomic number**

In [ ]:
import matplotlib.pyplot as plt

# Cell 06

plt.figure(figsize=(20, 8))
x = np.arange(len(elements))
plt.plot(x, melt, color="turquoise", marker=".", label="Melting Point")
plt.plot(x, boil, color="coral", marker=".", label="Boiling Point")
plt.title("Melting and Boiling Point")
plt.xlabel("Elements (By Group, Period, Atomic Number)")
plt.ylabel("Temperature (C)")
ax = plt.gca()
ax.set_xticks(x)
ax.set_xticklabels(data[:, 0], fontsize=9, rotation=90)
ax.legend(loc="lower center")
ax.grid(True)
plt.show()